# 第三板块衍生：creditos（clavePrevencion 预防类型）

本 Notebook 针对原始数据 `衍生原始数据_331条.csv` 中 `response_body.creditos` 做“预防类型（`clavePrevencion`）”维度的特征衍生。

## 你要的字段（平铺明细）

每条 creditos 账户记录平铺为一行（`apply_id` 重复），并保留：
- `apply_id`
- `request_time`（截止时间）
- `fechaAperturaCuenta`（开户日期，用来做时间窗口）
- `nombreOtorgante`（机构类别，明细里保留，便于你后续扩展）
- `clavePrevencion`（预防类型，下面要按这个做统计）
- `tipoResponsabilidad`（责任类型，下面也要按这个做统计：I/M/A）

## 预防类型口径（你给的 5 类）

- `CL`：账户已关闭，客户已全额支付且未造成违约，账户正常关闭
- `IA`：账户已停用，账户余额为零，处于非活动状态
- `LC`：与消费者达成减免或分期付款协议（协商还款安排）
- `PC`：账户进入催收
- `UP`：导致违约事件的账户

## 窗口与特征（大白话）

- **两个时间点**：
  - `request_time`（截止时间）
  - `fechaAperturaCuenta`（开户日期）
- **时间对比**：
  - `days_before_request_open = request_time - fechaAperturaCuenta`（换算成“天”，可带小数）
- **窗口筛选**：
  - 对每个窗口 N（30/60/90/120/180/360/720 天），只统计满足 `0 <= days_before_request_open <= N` 的 creditos 记录
- **输出特征（每个窗口 N）**：
  - `creditos_{N}d_total_cnt`：窗口内 creditos 总记录数
  - 对每个预防类型（CL/IA/LC/PC/UP）：
    - `creditos_{N}d_prev_{code}_cnt`：该类型数量
    - `creditos_{N}d_prev_{code}_ratio`：该类型数量 / 总记录数
  - 对每个责任类型（I/M/A）：
    - `creditos_{N}d_resp_{code}_cnt`：该类型数量
    - `creditos_{N}d_resp_{code}_ratio`：该类型数量 / 总记录数

## 异常兜底（A/C/D -> -999）

沿用前两块规则：
- A：关键字段缺失（apply_id / request_time / response_body）
- C：response_body 为空或非法 JSON
- D：response_body 里没有 creditos（或 creditos 不是 list）

命中时：该 apply_id 的**所有衍生特征**统一填 `-999`。

## 输出（默认不写，等你确认后再写）

- `outputs/clave_prevencion_flat.csv`：平铺明细
- `outputs/features_clave_prevencion.csv`：衍生特征表


In [ ]:
# 大白话：这一格只做“读数据+把截止时间 request_time 统一解析成时间类型+准备 A/C/D 兜底标记”，不做复杂衍生。

# ========================= 这套衍生“到底衍生了什么”（大白话总说明）=========================
# 1) 一共衍生多少个特征？
#    这一板块只做两套“类别计数/占比”衍生（不做金额均值/标准差那种复杂统计）：
#    - clavePrevencion（预防类型）：5 类（CL/IA/LC/PC/UP）
#    - tipoResponsabilidad（责任类型）：3 类（I/M/A）
#    - 时间窗口：7 个（30/60/90/120/180/360/720 天）
#
#    每个窗口 N 天内输出：
#    - 1 个：creditos_{N}d_total_cnt（窗口内 creditos 总记录数）
#    - 预防类型：5 类 × 2 个（cnt+ratio）= 10 个
#    - 责任类型：3 类 × 2 个（cnt+ratio）= 6 个
#    所以：
#    - 每窗口特征数 = 1 + 10 + 6 = 17
#    - 总特征数 = 17 × 7 = 119
#    另外：特征表里会保留 request_time（原始截止时间，不算衍生特征）。
#
# 2) 分哪些板块？每块在算什么？
#    - [板块A：总量]：窗口内 creditos 总记录数（总共有多少条账户记录落入窗口）
#    - [板块B：预防类型画像]：窗口内 CL/IA/LC/PC/UP 各有多少条（cnt）+ 各自占总记录的比例（ratio）
#    - [板块C：责任类型画像]：窗口内 I/M/A 各有多少条（cnt）+ 各自占总记录的比例（ratio）
#
# 3) 核心逻辑（窗口/分组/统计口径）
#    - 两个时间点：
#      * request_time：每个 apply_id 自己的截止时间
#      * fechaAperturaCuenta：每条 creditos 账户的开户日期（用来做窗口筛选）
#    - 时间对比怎么做：
#      * 先用 pd.to_datetime(..., errors='coerce') 统一解析日期
#      * 相减：request_time - fechaAperturaCuenta 得到 Timedelta
#      * 转天：timedelta.dt.total_seconds() / 86400 得到 days_before_request_open（可带小数）
#    - 窗口筛选：0 <= days_before_request_open <= N
#    - 分组维度：按 apply_id 聚合；在每个 apply_id 内再按类别（预防类型/责任类型）计数
#    - 统计口径：
#      * cnt：窗口内该类别条数
#      * ratio：cnt / creditos_{N}d_total_cnt（分母为 0 时 ratio 记为 0）
#
# 4) 缺失/异常怎么处理？你最后会看到 0 / NaN / -999 三种表现
#    - A/C/D（硬规则）：
#      * A：关键字段缺失（apply_id/request_time/response_body）
#      * C：response_body 为空/非法 JSON
#      * D：response_body 里没有 creditos（或 creditos 不是 list）
#      => 命中时：该 apply_id 的所有衍生特征统一 -999。
#    - creditos 存在但为空列表 []：不算 D，只是没有账户记录
#      => 该窗口 total_cnt=0、所有类别 cnt=0、所有 ratio=0。
#    - fechaAperturaCuenta 解析失败或未来开户（>request_time）：
#      => 这条明细会被过滤掉，不参与任何窗口统计。
# =============================================================================

# 代码格 1/4：读取数据（逐行注释版）

# 大白话：导入 json，用来解析 response_body 里的 JSON 字符串。
import json
# 大白话：导入 Path，用来更稳地处理文件路径。
from pathlib import Path

# 大白话：导入 numpy，用来处理 NaN、做数值计算。
import numpy as np
# 大白话：导入 pandas，用来读 CSV、处理表格数据、做聚合。
import pandas as pd

# 大白话：为了让所有板块口径一致，这里优先读取你筛选后的 pickle：cdc_pickle_pass_fpd7.pkl。
# 大白话：如果你想回到 331 条 CSV，只需要把 USE_PICKLE 改成 False。
USE_PICKLE = True  # True=优先读 pickle；False=读 CSV
INPUT_PICKLE = Path("cdc_pickle_pass_fpd7.pkl")  # 你筛选后的数据

# 大白话：定义输入 CSV 文件路径（兜底：pickle 不存在才用它）。
INPUT_CSV = Path("衍生原始数据_331条.csv")
# 大白话：定义输出目录（默认不写，等你确认后再开开关）。
OUTPUT_DIR = Path("outputs")
# 大白话：确保 outputs 目录存在（没有就创建）。
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 大白话：读入数据（优先 pickle；否则读 CSV）。
if USE_PICKLE and INPUT_PICKLE.exists():
    df_raw = pd.read_pickle(INPUT_PICKLE)
    print("[INFO] loaded pickle:", INPUT_PICKLE)
else:
    df_raw = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    print("[INFO] loaded csv:", INPUT_CSV)

# 大白话：cdc_pickle_pass_fpd7.pkl 里一般用 apply_time 表示“截止日期”，这里统一映射成 request_time。
if ("request_time" not in df_raw.columns) and ("apply_time" in df_raw.columns):
    df_raw["request_time"] = df_raw["apply_time"]
    print("[INFO] mapped apply_time -> request_time")

# 大白话：把截止日期统一解析成 pandas datetime（解析失败 -> NaT）。
df_raw["request_time"] = pd.to_datetime(df_raw.get("request_time"), errors="coerce")

# 大白话：定义本板块最关键的三列（缺任何一个都属于情况 A）。
required_cols = ["apply_id", "request_time", "response_body"]
# 大白话：找出哪些关键列缺失。
missing = [c for c in required_cols if c not in df_raw.columns]

# 大白话：情况 A 判定：只要缺关键列就记为 True。
FATAL_MISSING_COLUMNS = len(missing) > 0

# 大白话：如果命中情况 A，就打印提示并补齐缺失列，保证后续还能跑出 -999 特征。
if FATAL_MISSING_COLUMNS:
    # 大白话：提示你哪些字段缺失了。
    print("[WARN] 命中情况A：关键字段缺失 -> 后续该申请的衍生特征将统一填 -999")
    # 大白话：打印具体缺失列名。
    print("[WARN] 缺失字段:", missing)

    # 大白话：补齐 apply_id 列（如果缺了）。
    if "apply_id" not in df_raw.columns:
        df_raw["apply_id"] = pd.Series(dtype="int64")
    # 大白话：补齐 request_time 列（如果缺了）。
    if "request_time" not in df_raw.columns:
        df_raw["request_time"] = pd.NaT
    # 大白话：补齐 response_body 列（如果缺了）。
    if "response_body" not in df_raw.columns:
        df_raw["response_body"] = None

# 大白话：把 request_time 统一解析成 datetime（解析失败变 NaT，不中断）。
df_raw["request_time"] = pd.to_datetime(df_raw["request_time"], errors="coerce")

# 大白话：打印原始数据规模，方便你核对。
print("df_raw shape:", df_raw.shape)
# 大白话：展示前几行快速确认字段内容。
df_raw.head(3)


df_raw shape: (331, 10)


,apply_id,response_body,request_time,approve_state,credit_limit_amount,use_amount,principal_amount_borrowed,fpd7,spd7,blind_lend
0,1065991091661283329,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 01:53:44.943,CYCLE_PASS,800.0,700.0,700.0,0,0,NaN
1,1066560157648134145,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 06:29:46.980,CYCLE_PASS,500.0,500.0,500.0,0,0,1.0
2,1066719243236777985,"{""claveOtorgante"":""004676"",""consultas"":[{""clav...",2025-11-25 07:47:05.929,SINGLE_PASS,500.0,500.0,500.0,0,0,NaN


In [2]:
# 大白话：这一格负责把 response_body.creditos 解析并平铺成明细表（apply_id 重复），再把开户日期转成时间并计算“距截止日多少天”，给后面窗口统计用。

# 代码格 2/4：处理数据（逐行注释版）

# 大白话：定义我们要统计的预防类型类别（你指定的 5 类）。
CLAVE_PREV_TO_USE = ["CL", "IA", "LC", "PC", "UP"]
# 大白话：为了列名更稳定，我们把类别 code 映射成小写 id。
CLAVE_PREV_TO_ID = {c: c.lower() for c in CLAVE_PREV_TO_USE}

# 大白话：定义我们要统计的责任类型类别（你指定的 3 类）。
RESP_TO_USE = ["I", "M", "A"]
# 大白话：为了列名更稳定，我们把责任类型 code 映射成小写 id。
RESP_TO_ID = {c: c.lower() for c in RESP_TO_USE}

# 大白话：准备一个列表，用来装平铺后的每一行 creditos 明细。
rows = []

# 大白话：为每个 apply_id 建一个状态字典，默认 ok；如果命中 A/C/D，后面整行特征要置 -999。
apply_status = {aid: "ok" for aid in df_raw["apply_id"].tolist()}

# 大白话：如果第 1 格已经判定命中情况 A，则把所有 apply_id 标记成 A。
if "FATAL_MISSING_COLUMNS" in globals() and FATAL_MISSING_COLUMNS:
    for aid in apply_status:
        apply_status[aid] = "A_missing_columns"

# 大白话：逐行遍历原始表：每行对应一个申请（apply_id）及其截止时间与 JSON。
for apply_id, request_time, response_body in df_raw[["apply_id", "request_time", "response_body"]].itertuples(index=False, name=None):
    # 大白话：情况 C：response_body 缺失/空字符串，直接记为 bad_json。
    if response_body is None or (isinstance(response_body, float) and pd.isna(response_body)) or str(response_body).strip() == "":
        apply_status[apply_id] = "C_bad_json"
        continue

    # 大白话：尝试解析 JSON；失败也属于情况 C。
    try:
        obj = json.loads(response_body) if isinstance(response_body, str) else response_body
    except Exception:
        apply_status[apply_id] = "C_bad_json"
        continue

    # 大白话：情况 D：没有 creditos 块（或者结构不是 dict）。
    if not isinstance(obj, dict) or ("creditos" not in obj):
        apply_status[apply_id] = "D_no_creditos_key"
        continue

    # 大白话：取出 creditos；必须是 list 才算结构正确。
    creditos = obj.get("creditos")
    if not isinstance(creditos, list):
        apply_status[apply_id] = "D_no_creditos_key"
        continue

    # 大白话：正常情况：creditos 是 list（可以为空 list，代表没有账户记录，不属于 D）。
    for it in creditos:
        if not isinstance(it, dict):
            continue

        # 大白话：把你要的字段从单条账户记录里取出来，平铺成一行。
        rows.append(
            {
                "apply_id": apply_id,
                "request_time": request_time,
                "fechaAperturaCuenta": it.get("fechaAperturaCuenta"),
                "nombreOtorgante": it.get("nombreOtorgante"),
                "clavePrevencion": it.get("clavePrevencion"),
                "tipoResponsabilidad": it.get("tipoResponsabilidad"),
            }
        )

# 大白话：把 rows 变成 DataFrame，就是平铺明细表。
flat_df = pd.DataFrame(rows)

# 大白话：统一解析开户日期字段（解析失败变 NaT）。
flat_df["fechaAperturaCuenta"] = pd.to_datetime(flat_df["fechaAperturaCuenta"], errors="coerce")

# 大白话：把 clavePrevencion 标准化成大写字符串（缺失变空字符串），方便后面 isin 统计。
flat_df["clave_prevencion"] = flat_df["clavePrevencion"].fillna("").astype(str).str.strip().str.upper()

# 大白话：把 tipoResponsabilidad 标准化成大写字符串（缺失变空字符串），方便后面 isin 统计。
flat_df["tipo_responsabilidad"] = flat_df["tipoResponsabilidad"].fillna("").astype(str).str.strip().str.upper()

# 大白话：计算时间差：request_time - 开户日期，转成“天”（可能带小数）。
flat_df["days_before_request_open"] = (
    (flat_df["request_time"] - flat_df["fechaAperturaCuenta"]).dt.total_seconds() / 86400
)

# 大白话：过滤掉无法用于窗口的记录：开户日期解析失败或未来开户（负数）。
flat_df = flat_df[
    flat_df["days_before_request_open"].notna() & (flat_df["days_before_request_open"] >= 0)
].copy()

# 大白话：打印平铺明细规模，方便你核对。
print("flat_df shape:", flat_df.shape)
flat_df.head(5)


flat_df shape: (19233, 9)


,apply_id,request_time,fechaAperturaCuenta,nombreOtorgante,clavePrevencion,tipoResponsabilidad,clave_prevencion,tipo_responsabilidad,days_before_request_open
0,1065991091661283329,2025-11-25 01:53:44.943,2023-01-26,MERCANCIA PARA HOGAR Y OFICINA,None,I,,I,1034.078992
1,1065991091661283329,2025-11-25 01:53:44.943,2023-12-25,MERCANCIA PARA HOGAR Y OFICINA,CC,I,CC,I,701.078992
2,1065991091661283329,2025-11-25 01:53:44.943,2025-09-09,MERCANCIA PARA HOGAR Y OFICINA,None,I,,I,77.078992
3,1065991091661283329,2025-11-25 01:53:44.943,2023-02-12,MERCANCIA PARA HOGAR Y OFICINA,None,I,,I,1017.078992
4,1065991091661283329,2025-11-25 01:53:44.943,2025-02-04,SOCIEDAD FINANCIERA DE OBJETO MULTIPLE,None,I,,I,294.078992


In [3]:
# 大白话：这一格把平铺明细 flat_df 按窗口聚合成特征表：对 CL/IA/LC/PC/UP 分别算数量和占比，并给 A/C/D 的申请统一置 -999。

# 代码格 3/4：衍生函数逻辑（逐行注释版）

# 大白话：定义主衍生函数，输入原始表与平铺明细，输出按 apply_id 聚合的特征表。
def derive_prevencion_features(
    df_raw: pd.DataFrame,
    flat_df: pd.DataFrame,
    window_days_list: list[int],
    prev_to_use: list[str],
    prev_to_id: dict[str, str],
    resp_to_use: list[str],
    resp_to_id: dict[str, str],
    apply_status: dict | None = None,
    sentinel_value: float = -999,
) -> pd.DataFrame:
    """按 request_time 截止日，对 creditos 的 clavePrevencion + tipoResponsabilidad 做多窗口计数/占比特征。"""

    # 大白话：构造基表：每个 apply_id 一行，保留 request_time（不是衍生特征，只是基准信息）。
    base = (
        df_raw[["apply_id", "request_time"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")
        .sort_index()
    )

    # 大白话：初始化特征表（后面不断拼接各窗口特征列）。
    feats = base.copy()

    # 大白话：对每个窗口 N 单独计算一批特征列，然后 join 到 feats。
    for w in window_days_list:
        # 大白话：筛选窗口内明细：0<=days_before_request_open<=w。
        sub = flat_df[(flat_df["days_before_request_open"] >= 0) & (flat_df["days_before_request_open"] <= w)].copy()

        # 大白话：先建一个只含索引的输出表，用来装该窗口的所有特征。
        out = pd.DataFrame(index=base.index)

        # 大白话：窗口内总记录数（分母）：窗口内所有 creditos 明细行数。
        total = sub.groupby("apply_id").size().reindex(base.index).fillna(0).astype(int)
        out[f"creditos_{w}d_total_cnt"] = total

        # 大白话：只统计你关注的 5 个预防类型。
        sub_prev = sub[sub["clave_prevencion"].isin(prev_to_use)].copy()

        # 大白话：按 apply_id+预防类型计数，展开成列，对齐到所有 apply_id 与所有类别。
        prev_cnt = (
            sub_prev.groupby(["apply_id", "clave_prevencion"]).size().unstack().reindex(index=base.index, columns=prev_to_use)
        )
        prev_cnt = prev_cnt.fillna(0).astype(int)

        # 大白话：占比 = 该类数量 / total_cnt；total=0 时占比按 0。
        prev_ratio = prev_cnt.div(total.replace(0, np.nan), axis=0).fillna(0.0)

        # 大白话：把每个预防类型的 cnt/ratio 写成独立特征列。
        for code in prev_to_use:
            cid = prev_to_id.get(code, code.lower())
            out[f"creditos_{w}d_prev_{cid}_cnt"] = prev_cnt[code]
            out[f"creditos_{w}d_prev_{cid}_ratio"] = prev_ratio[code]

        # 大白话：只统计你关注的 3 个责任类型（I/M/A）。
        sub_resp = sub[sub["tipo_responsabilidad"].isin(resp_to_use)].copy()

        # 大白话：按 apply_id+责任类型计数，展开成列，对齐到所有 apply_id 与所有类别。
        resp_cnt = (
            sub_resp.groupby(["apply_id", "tipo_responsabilidad"]).size().unstack().reindex(index=base.index, columns=resp_to_use)
        )
        resp_cnt = resp_cnt.fillna(0).astype(int)

        # 大白话：占比 = 该类数量 / total_cnt；total=0 时占比按 0。
        resp_ratio = resp_cnt.div(total.replace(0, np.nan), axis=0).fillna(0.0)

        # 大白话：把每个责任类型的 cnt/ratio 写成独立特征列。
        for code in resp_to_use:
            cid = resp_to_id.get(code, code.lower())
            out[f"creditos_{w}d_resp_{cid}_cnt"] = resp_cnt[code]
            out[f"creditos_{w}d_resp_{cid}_ratio"] = resp_ratio[code]

        # 大白话：把该窗口的特征拼到总表里。
        feats = feats.join(out, how="left")

    # 大白话：按你规则处理 A/C/D：只覆盖衍生特征列，不覆盖 request_time。
    if apply_status is not None:
        bad_ids = [aid for aid, st in apply_status.items() if st in {"A_missing_columns", "C_bad_json", "D_no_creditos_key"}]
        if len(bad_ids) > 0:
            feature_cols = [c for c in feats.columns if c != "request_time"]
            feats.loc[bad_ids, feature_cols] = sentinel_value

    # 大白话：返回特征表（index=apply_id）。
    return feats


In [ ]:
# 大白话：这一格调用上一格的函数真正生成特征表 features，并用 WRITE_OUTPUTS 控制是否写出两份 CSV（默认不写，等你确认）。

# 代码格 4/4：生成结果（逐行注释版）

# 大白话：定义你要的时间窗口列表。
WINDOW_DAYS = [30, 60, 90, 120, 180, 360, 720]

# 大白话：调用衍生函数生成特征表。
features = derive_prevencion_features(
    df_raw=df_raw,
    flat_df=flat_df,
    window_days_list=WINDOW_DAYS,
    prev_to_use=CLAVE_PREV_TO_USE,
    prev_to_id=CLAVE_PREV_TO_ID,
    resp_to_use=RESP_TO_USE,
    resp_to_id=RESP_TO_ID,
    apply_status=apply_status,
    sentinel_value=-999,
)

# 大白话：输出开关（默认 False，不写文件；你确认后再改 True）。
WRITE_OUTPUTS = True

# 大白话：定义两份输出文件路径（明细 + 特征）。
flat_out_path = OUTPUT_DIR / "clave_prevencion_flat.csv"
feat_out_path = OUTPUT_DIR / "features_clave_prevencion.csv"

# 大白话：只有你确认后才真正写出 outputs 文件。
if WRITE_OUTPUTS:
    flat_df.to_csv(flat_out_path, index=False, encoding="utf-8-sig")

    # 只对“衍生特征列”加前缀/后缀（request_time 不加），并写出到 CSV
    _rename_map_for_write = {c: f"cdc_{c}_v2" for c in features.columns if c != "request_time"}

    _features_to_write = features.rename(columns=_rename_map_for_write).reset_index()  # apply_id 变回普通列，便于写 CSV
    _round_cols = [c for c in _features_to_write.columns if c not in {"apply_id", "request_time"}]  # 只对衍生特征列做保留2位小数
    _features_to_write[_round_cols] = _features_to_write[_round_cols].apply(pd.to_numeric, errors="coerce").round(2)  # 统一保留小数点后2位

    _features_to_write.to_csv(
        feat_out_path,
        index=False,
        encoding="utf-8-sig",
    )

    print("written:")
    print("-", flat_out_path)
    print("-", feat_out_path)
else:
    print("[INFO] 当前 WRITE_OUTPUTS=False：已计算完成，但未写出 outputs/*.csv（等你确认后再输出）")

# 大白话：打印一下结果规模，方便你核对。
print("flat rows:", len(flat_df))
print("features shape:", features.shape)

# === 特征数据字典（不落盘，仅用于核对/拼接）===
# 大白话：这里生成第三板块的 feature_dict，并且字段格式与前两块一致，可直接上下拼接。

import re


def build_feature_dict_block3(cols: list[str]) -> pd.DataFrame:
    rows = []

    for col in cols:
        if col == "request_time":
            continue

        m = re.match(r"^creditos_(\d+)d_total_cnt$", col)
        if m:
            w = int(m.group(1))
            rows.append(
                {
                    "block": "block3_creditos_prev_resp",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "total",
                    "group_code": "ALL",
                    "metric": "count",
                    "stat": "cnt",
                    "description": f"第三板块：近{w}天内 creditos 总记录数",
                }
            )
            continue

        m = re.match(r"^creditos_(\d+)d_prev_([a-z]+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            k = m.group(3)
            rows.append(
                {
                    "block": "block3_creditos_prev_resp",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "clave_prevencion",
                    "group_code": code,
                    "metric": "count",
                    "stat": k,
                    "description": f"第三板块：近{w}天内 clavePrevencion={code} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                }
            )
            continue

        m = re.match(r"^creditos_(\d+)d_resp_([a-z]+)_(cnt|ratio)$", col)
        if m:
            w = int(m.group(1))
            code = m.group(2).upper()
            k = m.group(3)
            rows.append(
                {
                    "block": "block3_creditos_prev_resp",
                    "feature_name": col,
                    "window_days": w,
                    "group_type": "tipo_responsabilidad",
                    "group_code": code,
                    "metric": "count",
                    "stat": k,
                    "description": f"第三板块：近{w}天内 tipoResponsabilidad={code} 的{'数量' if k=='cnt' else '占比（该类/总记录数）'}",
                }
            )
            continue

        rows.append(
            {
                "block": "block3_creditos_prev_resp",
                "feature_name": col,
                "window_days": None,
                "group_type": "unknown",
                "group_code": None,
                "metric": None,
                "stat": None,
                "description": "第三板块：未匹配到解析规则（请人工确认列名口径）",
            }
        )

    return pd.DataFrame(
        rows,
        columns=[
            "block",
            "feature_name",
            "window_days",
            "group_type",
            "group_code",
            "metric",
            "stat",
            "description",
        ],
    )


feature_dict = build_feature_dict_block3(list(features.columns))
print("feature_dict shape:", feature_dict.shape)
feature_dict.head(10)

# === 导出：特征字典 CSV（3列：特征名称/中文释义/逻辑解释）===
# 大白话：你要的是每个板块各自导出一份“特征字典CSV”，用于你后续手工上下拼接。
# 大白话：默认不写文件：你确认后把 WRITE_FEATURE_DICT_OUTPUTS 改成 True 才会写出。
WRITE_FEATURE_DICT_OUTPUTS = False

# 大白话：第三板块涉及的原始字段释义（用于 logic_desc 中解释字段名）。
RAW_FIELD_DESC_BLOCK3 = {
    "response_body.creditos": "信贷记录列表",
    "fechaAperturaCuenta": "账户开立日期（开户日期）",
    "clavePrevencion": "预防/风险提示代码（CL/IA/LC/PC/UP）",
    "tipoResponsabilidad": "责任类型（I个人/M联名/A授权等）",
    "request_time": "截止日期（本次申请时间）",
}

STAT_DESC_BLOCK3 = {
    "cnt": "数量（窗口内该组记录数）",
    "ratio": "占比=该组cnt/窗口内总记录数",
}

def _logic_desc_block3(r) -> str:
    w = r.get("window_days")
    gtype = r.get("group_type")
    gcode = r.get("group_code")
    metric = r.get("metric")
    stat = r.get("stat")

    window_desc = (
        f"窗口={w}天（基于fechaAperturaCuenta=账户开立日期，与request_time=截止日期计算days_before_request_open=(request_time-fechaAperturaCuenta)/86400，再取0<=days<=window）"
    )
    group_desc = f"分组={gtype}:{gcode}（来源字段：clavePrevencion=风险提示代码 / tipoResponsabilidad=责任类型）"
    stat_explain = STAT_DESC_BLOCK3.get(str(stat), str(stat))
    metric_desc = f"指标={metric}; 统计={stat}（{stat_explain}）"
    raw_fields = (
        "原始字段释义："
        "response_body.creditos=信贷记录列表；"
        "fechaAperturaCuenta=账户开立日期；"
        "clavePrevencion=预防/风险提示代码；"
        "tipoResponsabilidad=责任类型；"
        "request_time=截止日期"
    )
    return f"{window_desc}; {group_desc}; {metric_desc}; {raw_fields}"

feature_dict_3col = pd.DataFrame(
    {
        "feature_name": feature_dict["feature_name"],
        "cn_desc": feature_dict["description"],
        "logic_desc": feature_dict.apply(_logic_desc_block3, axis=1),
    }
)

feature_dict_out_path = OUTPUT_DIR / "feature_dict_prev_resp.csv"

if WRITE_FEATURE_DICT_OUTPUTS:
    feature_dict_3col.to_csv(feature_dict_out_path, index=False, encoding="utf-8-sig")
    print("written feature_dict:")
    print("-", feature_dict_out_path)
else:
    print("[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出特征字典CSV（等你确认后再输出）")

# === 特征列名统一加前后缀（只改 features，不改特征字典）===
FEATURE_PREFIX = "cdc_"
FEATURE_SUFFIX = "_v2"

rename_map = {
    c: f"{FEATURE_PREFIX}{c}{FEATURE_SUFFIX}"
    for c in features.columns
    if c != "request_time"
}
features = features.rename(columns=rename_map)

# === 衍生特征统一保留2位小数（只改 features，不改特征字典；request_time 不动）===
_round_cols = [c for c in features.columns if c != "request_time"]
features[_round_cols] = features[_round_cols].apply(pd.to_numeric, errors="coerce").round(2)

features.head(3)


[INFO] 当前 WRITE_OUTPUTS=False：已计算完成，但未写出 outputs/*.csv（等你确认后再输出）
flat rows: 19233
features shape: (331, 120)
feature_dict shape: (119, 8)
[INFO] 当前 WRITE_FEATURE_DICT_OUTPUTS=False：未写出特征字典CSV（等你确认后再输出）


,request_time,cdc_creditos_30d_total_cnt_v2,cdc_creditos_30d_prev_cl_cnt_v2,cdc_creditos_30d_prev_cl_ratio_v2,cdc_creditos_30d_prev_ia_cnt_v2,cdc_creditos_30d_prev_ia_ratio_v2,cdc_creditos_30d_prev_lc_cnt_v2,cdc_creditos_30d_prev_lc_ratio_v2,cdc_creditos_30d_prev_pc_cnt_v2,cdc_creditos_30d_prev_pc_ratio_v2,...,cdc_creditos_720d_prev_pc_cnt_v2,cdc_creditos_720d_prev_pc_ratio_v2,cdc_creditos_720d_prev_up_cnt_v2,cdc_creditos_720d_prev_up_ratio_v2,cdc_creditos_720d_resp_i_cnt_v2,cdc_creditos_720d_resp_i_ratio_v2,cdc_creditos_720d_resp_m_cnt_v2,cdc_creditos_720d_resp_m_ratio_v2,cdc_creditos_720d_resp_a_cnt_v2,cdc_creditos_720d_resp_a_ratio_v2
apply_id,,,,,,,,,,,,,,,,,,,,,
1065543521709277185,2025-11-24 22:16:38.276,0,0,0.0,0,0.0,0,0.0,0,0.0,...,0,0.00,0,0.0,14,1.0,0,0.0,0,0.0
1065991091661283329,2025-11-25 01:53:44.943,0,0,0.0,0,0.0,0,0.0,0,0.0,...,1,0.01,0,0.0,94,1.0,0,0.0,0,0.0
1066560157648134145,2025-11-25 06:29:46.980,0,0,0.0,0,0.0,0,0.0,0,0.0,...,0,0.00,0,0.0,5,1.0,0,0.0,0,0.0


In [ ]:
# TODO（衍生报告：把评估结果写进一个 Excel，多 sheet）
# 大白话：你确认了“评估结果要写进一个 Excel 作为衍生报告”，所以这一格会把本 notebook 的评估输出落盘。
# 说明：
# - 如果 df_raw 里有 fpd7/approve_state：会额外产出分周坏率、IV、PSI、TopIV 分箱、分周IV。
# - 如果没有标签列：只输出特征画像（空值率/同一值率）与元信息。

import re  # 用于清洗 sheet 名
from pathlib import Path  # 用于拼接输出路径

import numpy as np  # 数值计算
import pandas as pd  # 表格计算

# 大白话：写报告开关（你已明确要写，所以默认 True）。
WRITE_REPORTS = True

# 大白话：输出文件路径（第三板块报告）。
REPORT_XLSX = Path("outputs") / "block3_eval_report.xlsx"

# 大白话：统一把 NaN 或 -999 当作空值（-999 是 A/C/D 的兜底值）。
SENTINEL_VALUE = -999

# 大白话：Excel 的 sheet 名最长 31 且不能重复，这里做一个安全命名。
def _safe_sheet_name(name: str, used: set[str]) -> str:
    base = re.sub(r"[^0-9a-zA-Z_]+", "_", str(name))[:31] or "sheet"  # 非法字符替换成下划线
    s = base  # 初始候选
    i = 1  # 冲突计数器
    while s in used:  # 如果重名就加后缀
        suf = f"_{i}"  # 后缀
        s = base[: max(0, 31 - len(suf))] + suf  # 保证仍<=31
        i += 1  # 递增
    used.add(s)  # 记录
    return s  # 返回安全 sheet 名

# 大白话：拿到原始数据与特征表（本 notebook 默认变量名是 df_raw / features）。
_raw = df_raw if "df_raw" in globals() else None  # 原始表
_feat = features if "features" in globals() else None  # 特征表

if _raw is None or _feat is None:
    raise RuntimeError("[REPORT] 未找到 df_raw 或 features，请先从头运行 notebook 生成它们。")

# 大白话：把 apply_id 作为索引，方便对齐。
_feat2 = _feat.copy()  # 复制，避免污染上游变量
if "apply_id" in _feat2.columns:
    _feat2 = _feat2.set_index("apply_id")  # 把 apply_id 设为索引

# 大白话：确保 request_time 是 datetime（用于分周）。
if "request_time" in _feat2.columns:
    _feat2["request_time"] = pd.to_datetime(_feat2["request_time"], errors="coerce")

# 大白话：X 是所有衍生特征列（不含 request_time）。
X = _feat2.drop(columns=["request_time"], errors="ignore")

# 大白话：空值掩码（NaN 或 -999）。
missing_mask = X.isna() | (X == SENTINEL_VALUE)

# 大白话：空值率（每列空值占比）。
na_rate = missing_mask.mean(axis=0)

# 大白话：同一值率（出现频率最高的值占比；空值统一当作同一个值来算）。
_same = {}
for c in X.columns:
    s = X[c].copy()
    s = s.mask(missing_mask[c], "__MISSING__")
    vc = s.value_counts(dropna=False, normalize=True)
    _same[c] = float(vc.iloc[0]) if len(vc) > 0 else 1.0
same_value_rate = pd.Series(_same).reindex(X.columns)

# 大白话：如果有标签 fpd7，就做更完整的评估（IV/分周坏率/PSI）。
has_fpd7 = "fpd7" in _raw.columns
has_state = "approve_state" in _raw.columns

# 大白话：对齐 y（坏=1 好=0）。
if has_fpd7:
    y = (
        _raw[["apply_id", "fpd7"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")["fpd7"]
        .pipe(pd.to_numeric, errors="coerce")
        .reindex(_feat2.index)
    )
    y01 = (y > 0).astype("int8")
else:
    y = None
    y01 = None

# 大白话：对齐 approve_state（cycle_pass / single_pass）。
if has_state:
    state = (
        _raw[["apply_id", "approve_state"]]
        .drop_duplicates("apply_id")
        .set_index("apply_id")["approve_state"]
        .fillna("")
        .astype(str)
        .str.lower()
        .reindex(_feat2.index)
        .fillna("")
    )
    mask_cycle = state.str.contains("cycle_pass", na=False)
    mask_single = state.str.contains("single_pass", na=False)
else:
    mask_cycle = pd.Series(False, index=_feat2.index)
    mask_single = pd.Series(False, index=_feat2.index)

mask_all = pd.Series(True, index=_feat2.index)

# 大白话：分周（周一 00:00:00）。
if "request_time" in _feat2.columns:
    t = _feat2["request_time"]
    week_start = (t - pd.to_timedelta(t.dt.weekday, unit="D")).dt.normalize()
else:
    week_start = pd.Series(pd.NaT, index=_feat2.index)

# ========== 可写入 Excel 的表集合 ==========
TABLES: dict[str, pd.DataFrame] = {}

# 1) 特征画像（空值率/同一值率）
profile_df = (
    pd.DataFrame({"feature": X.columns, "na_rate": na_rate.values, "same_value_rate": same_value_rate.values})
    .sort_values(["na_rate", "same_value_rate"], ascending=[False, False])
    .reset_index(drop=True)
)
TABLES["feature_profile"] = profile_df

# 2) 分周坏率（需要 y01）
if y01 is not None:
    def _weekly_bad_rate(mask: pd.Series, name: str) -> pd.DataFrame:
        idx = _feat2.index[mask]
        tmp = pd.DataFrame({"week": week_start.reindex(idx), "y": y01.reindex(idx)}).dropna(subset=["week", "y"])
        out = tmp.groupby("week", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        out["sample"] = name
        return out

    weekly_bad = pd.concat(
        [
            _weekly_bad_rate(mask_all, "overall"),
            _weekly_bad_rate(mask_cycle, "cycle_pass"),
            _weekly_bad_rate(mask_single, "single_pass"),
        ],
        axis=0,
        ignore_index=True,
    ).sort_values(["week", "sample"]).reset_index(drop=True)

    TABLES["weekly_bad_rate"] = weekly_bad

# 3) IV / PSI / TopIV 分箱 / 分周 IV（需要 y01）
if y01 is not None:
    IV_Q = 10  # IV 分箱：分位数箱数
    BIN_N = 5  # 报告分箱：5箱

    def _compute_iv_one(x: pd.Series, ybin: pd.Series) -> float:
        dfv = pd.DataFrame({"x": x, "y": ybin}).dropna(subset=["y"])
        if dfv.shape[0] == 0:
            return float("nan")
        x_raw = dfv["x"]
        yb = dfv["y"].astype(int)
        miss = x_raw.isna() | (x_raw == SENTINEL_VALUE)
        x_non = x_raw[~miss]
        if x_non.nunique(dropna=True) <= 2:
            b = x_raw.astype(object).where(~miss, "MISSING")
        else:
            try:
                b_non = pd.qcut(x_non, q=IV_Q, duplicates="drop")
                b = pd.Series("MISSING", index=x_raw.index, dtype="object")
                b.loc[~miss] = b_non.astype(str)
            except Exception:
                b = x_raw.astype(object).where(~miss, "MISSING")
        grp = pd.DataFrame({"b": b, "y": yb}).groupby("b", observed=False)["y"].agg(["count", "sum"]).rename(columns={"sum": "bad"})
        grp["good"] = grp["count"] - grp["bad"]
        bt = grp["bad"].sum(); gt = grp["good"].sum()
        if bt == 0 or gt == 0:
            return 0.0
        k = grp.shape[0]
        grp["bad_dist"] = (grp["bad"] + 0.5) / (bt + 0.5 * k)
        grp["good_dist"] = (grp["good"] + 0.5) / (gt + 0.5 * k)
        woe = np.log(grp["bad_dist"] / grp["good_dist"])
        iv = ((grp["bad_dist"] - grp["good_dist"]) * woe).sum()
        return float(iv)

    eligible = (na_rate < 0.9) & (same_value_rate < 0.99)
    cols = X.columns[eligible].tolist()

    iv_overall = {c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce"), y01) for c in cols}
    iv_overall_s = pd.Series(iv_overall, name="iv_overall")

    iv_cycle_s = pd.Series({c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce")[mask_cycle], y01[mask_cycle]) for c in cols}, name="iv_cycle_pass") if has_state and int(mask_cycle.sum())>0 else pd.Series(dtype="float64", name="iv_cycle_pass")
    iv_single_s = pd.Series({c: _compute_iv_one(pd.to_numeric(X[c], errors="coerce")[mask_single], y01[mask_single]) for c in cols}, name="iv_single_pass") if has_state and int(mask_single.sum())>0 else pd.Series(dtype="float64", name="iv_single_pass")

    weeks = sorted([w for w in week_start.dropna().unique()])
    first_w = weeks[0] if len(weeks) >= 1 else None
    last_two = weeks[-2:] if len(weeks) >= 2 else weeks
    base_mask = (week_start == first_w) if first_w is not None else pd.Series(False, index=_feat2.index)
    comp_mask = week_start.isin(last_two) if len(last_two) > 0 else pd.Series(False, index=_feat2.index)

    PSI_Q = 10
    EPS = 1e-6

    def _psi_one(x: pd.Series) -> float:
        miss = x.isna() | (x == SENTINEL_VALUE)
        xb = x[base_mask]; xc = x[comp_mask]
        mb = miss[base_mask]; mc = miss[comp_mask]
        if xb.shape[0] == 0 or xc.shape[0] == 0:
            return float("nan")
        xb_non = xb[~mb]
        if xb_non.nunique(dropna=True) <= 2:
            bb = xb.astype(object).where(~mb, "MISSING")
            bc = xc.astype(object).where(~mc, "MISSING")
        else:
            try:
                _, edges = pd.qcut(xb_non, q=PSI_Q, retbins=True, duplicates="drop")
                edges = sorted(set(edges.tolist()))
                if len(edges) < 3:
                    bb = xb.astype(object).where(~mb, "MISSING")
                    bc = xc.astype(object).where(~mc, "MISSING")
                else:
                    bb_non = pd.cut(xb_non, bins=edges, include_lowest=True)
                    bc_non = pd.cut(xc[~mc], bins=edges, include_lowest=True)
                    bb = pd.Series("MISSING", index=xb.index, dtype="object"); bc = pd.Series("MISSING", index=xc.index, dtype="object")
                    bb.loc[~mb] = bb_non.astype(str); bc.loc[~mc] = bc_non.astype(str)
            except Exception:
                bb = xb.astype(object).where(~mb, "MISSING")
                bc = xc.astype(object).where(~mc, "MISSING")
        pb = bb.value_counts(normalize=True); pc = bc.value_counts(normalize=True)
        cats = list(pb.index.union(pc.index))
        psi = 0.0
        for k in cats:
            p = max(float(pb.get(k, 0.0)), EPS)
            q = max(float(pc.get(k, 0.0)), EPS)
            psi += (p - q) * float(np.log(p / q))
        return float(psi)

    fpd7_num = y.reindex(_feat2.index)

    iv_gt = iv_overall_s[iv_overall_s > 0.1].sort_values(ascending=False)
    rows = []
    for feat in iv_gt.index.tolist():
        x = pd.to_numeric(X[feat], errors="coerce")
        miss = x.isna() | (x == SENTINEL_VALUE)
        valid = (~miss) & fpd7_num.notna()
        corr = float(pd.Series(x[valid]).corr(fpd7_num[valid], method="pearson")) if int(valid.sum()) >= 3 else float("nan")
        rows.append(
            {
                "feature_name": feat,
                "iv": float(iv_gt.loc[feat]),
                "corr_fpd7": corr,
                "na_rate_na_or_-999": float(miss.mean()),
                "psi_last2w_vs_firstw": _psi_one(x) if (first_w is not None and len(last_two) > 0) else float("nan"),
                "iv_cycle_pass": float(iv_cycle_s.get(feat, float("nan"))),
                "iv_single_pass": float(iv_single_s.get(feat, float("nan"))),
            }
        )
    iv_report = pd.DataFrame(rows)
    for c in ["iv", "corr_fpd7", "na_rate_na_or_-999", "psi_last2w_vs_firstw", "iv_cycle_pass", "iv_single_pass"]:
        if c in iv_report.columns:
            iv_report[c] = pd.to_numeric(iv_report[c], errors="coerce").round(4)
    TABLES["iv_gt_0p1_report"] = iv_report

    top_overall = str(iv_overall_s.sort_values(ascending=False).index[0]) if len(iv_overall_s) > 0 else ""
    top_cycle = str(iv_cycle_s.sort_values(ascending=False).index[0]) if len(iv_cycle_s) > 0 else ""
    top_single = str(iv_single_s.sort_values(ascending=False).index[0]) if len(iv_single_s) > 0 else ""

    def _bin_report(feat: str, mask: pd.Series, method: str) -> pd.DataFrame:
        x = pd.to_numeric(X[feat], errors="coerce")[mask]
        yb = y01.reindex(X.index)[mask]
        miss = x.isna() | (x == SENTINEL_VALUE)
        x_non = x[~miss]
        if x_non.shape[0] == 0:
            lab = pd.Series("MISSING", index=x.index, dtype="object")
        else:
            if method == "qcut":
                r = x_non.rank(method="first")
                b_non = pd.qcut(r, q=BIN_N, duplicates="drop")
                lab = pd.Series("MISSING", index=x.index, dtype="object")
                lab.loc[~miss] = b_non.astype(str)
            else:
                b_non = pd.cut(x_non, bins=BIN_N, include_lowest=True)
                lab = pd.Series("MISSING", index=x.index, dtype="object")
                lab.loc[~miss] = b_non.astype(str)
        box = pd.DataFrame({"bin": lab, "y": yb}).dropna(subset=["y"])
        out = box.groupby("bin", observed=False)["y"].agg(total_cnt="size", bad_cnt="sum").reset_index()
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        uniq = pd.DataFrame({"bin": lab, "x": x.where(~miss, np.nan)}).groupby("bin", observed=False)["x"].nunique(dropna=True).rename("unique_cnt").reset_index()
        out = out.merge(uniq, on="bin", how="left")
        out["unique_cnt"] = out["unique_cnt"].fillna(0).astype(int)
        out.insert(0, "feature", feat)
        out.insert(1, "method", method)
        return out

    def _add_bins(tag: str, feat: str, mask: pd.Series):
        if not feat:
            return
        TABLES[f"bin_{tag}_qcut"] = _bin_report(feat, mask, "qcut")
        TABLES[f"bin_{tag}_cut"] = _bin_report(feat, mask, "cut")

    _add_bins("overall", top_overall, mask_all)
    _add_bins("cycle_pass", top_cycle, mask_cycle)
    _add_bins("single_pass", top_single, mask_single)

    def _weekly_iv(feat: str, mask: pd.Series) -> pd.DataFrame:
        idx = _feat2.index[mask]
        tmp = pd.DataFrame({"week": week_start.reindex(idx), "x": pd.to_numeric(X[feat], errors="coerce").reindex(idx), "y": y01.reindex(idx)}).dropna(subset=["week", "y"])
        out_rows = []
        for wk, g in tmp.groupby("week", observed=False):
            out_rows.append({"week": wk, "iv": _compute_iv_one(g["x"], g["y"]), "total_cnt": int(g.shape[0]), "bad_cnt": int(g["y"].sum())})
        out = pd.DataFrame(out_rows).sort_values("week")
        out["bad_rate"] = (out["bad_cnt"] / out["total_cnt"]).round(4)
        out["iv"] = pd.to_numeric(out["iv"], errors="coerce").round(4)
        return out

    for tag, feat in [("overall_top", top_overall), ("cycle_top", top_cycle), ("single_top", top_single)]:
        if not feat:
            continue
        TABLES[f"weekly_iv_{tag}_overall"] = _weekly_iv(feat, mask_all)
        TABLES[f"weekly_iv_{tag}_cycle_pass"] = _weekly_iv(feat, mask_cycle)
        TABLES[f"weekly_iv_{tag}_single_pass"] = _weekly_iv(feat, mask_single)

# 4) 元信息（永远写）
meta = pd.DataFrame(
    {
        "item": ["feature_cnt", "has_fpd7", "has_approve_state", "na_rate_ge_0.9_cnt", "same_value_rate_ge_0.99_cnt"],
        "value": [
            str(int(X.shape[1])),
            str(bool(has_fpd7)),
            str(bool(has_state)),
            str(int((na_rate >= 0.9).sum())),
            str(int((same_value_rate >= 0.99).sum())),
        ],
    }
)

# ========== 写 Excel（多 sheet） ==========
if WRITE_REPORTS:
    REPORT_XLSX.parent.mkdir(parents=True, exist_ok=True)
    used: set[str] = set()
    try:
        writer = pd.ExcelWriter(REPORT_XLSX, engine="openpyxl")
    except Exception:
        writer = pd.ExcelWriter(REPORT_XLSX)

    with writer:
        meta.to_excel(writer, sheet_name=_safe_sheet_name("meta", used), index=False)
        for name, df in TABLES.items():
            if df is None:
                df = pd.DataFrame()
            df.to_excel(writer, sheet_name=_safe_sheet_name(name, used), index=False)

    print("[WRITE_EXCEL] block3 eval report saved:", str(REPORT_XLSX.resolve()))

# =========================
# 额外：生成 quality 检测 Excel（每个特征一行）
# =========================
# 大白话：按你的要求，第三板块 notebook 也能独立输出 quality 表。

WRITE_QUALITY_EXCEL = False  # True 才会写出
QUALITY_XLSX = Path("outputs") / "block3_feature_quality.xlsx"

if WRITE_QUALITY_EXCEL:
    import scripts.build_all_blocks_feature_quality_excel as q

    _base = df_raw[["apply_id", "request_time", "fpd7", "approve_state"]].copy()
    _base["apply_id"] = _base["apply_id"].astype(str)
    _base = _base.drop_duplicates("apply_id").set_index("apply_id")

    _y = (pd.to_numeric(_base["fpd7"], errors="coerce") > 0).astype(int)
    _state = _base["approve_state"].fillna("").astype(str).str.lower()
    _is_cycle = _state.str.contains("cycle_pass", na=False)
    _is_single = _state.str.contains("single_pass", na=False)

    _rt = pd.to_datetime(_base["request_time"], errors="coerce")
    _week_start = (_rt - pd.to_timedelta(_rt.dt.weekday, unit="D")).dt.normalize()
    _weeks = sorted([w for w in _week_start.dropna().unique()])
    _first_w = _weeks[0] if len(_weeks) >= 1 else None
    _last_two = _weeks[-2:] if len(_weeks) >= 2 else _weeks
    _base_mask = (_week_start == _first_w) if _first_w is not None else pd.Series(False, index=_week_start.index)
    _comp_mask = _week_start.isin(_last_two) if len(_last_two) > 0 else pd.Series(False, index=_week_start.index)

    _feat = features.copy()
    if "apply_id" in _feat.columns:
        _feat = _feat.set_index("apply_id")
    _feat.index = _feat.index.astype(str)
    _X = _feat.drop(columns=["request_time"], errors="ignore")

    _dict = feature_dict_3col.copy()
    _dict["feature_name"] = _dict["feature_name"].astype(str)
    _dict = _dict.drop_duplicates("feature_name").set_index("feature_name")

    _rows = []
    _total = int(_X.shape[0])

    for _col in _X.columns:
        _col_name = str(_col)
        _base_name = q._strip_feature_name(_col_name)

        _x = pd.to_numeric(_X[_col], errors="coerce")
        _miss = _x.isna() | (_x == q.SENTINEL)
        _na_cnt = int(_miss.sum())
        _na_rate = float(_na_cnt / _total) if _total else float("nan")
        _non = _x[~_miss]
        _nunique = int(_non.nunique(dropna=True))

        _cn = str(_dict["cn_desc"].get(_base_name, "")) if _base_name in _dict.index else ""
        _logic = str(_dict["logic_desc"].get(_base_name, "")) if _base_name in _dict.index else ""

        _corr = float("nan")
        _valid = (~_miss) & _y.notna()
        if int(_valid.sum()) >= 3:
            _corr = q._corr_pearson(_x[_valid].to_numpy(dtype="float64", copy=False), _y[_valid].to_numpy(dtype="float64", copy=False))

        _iv = q._iv_one(_x, _y)
        _psi = q._psi_one(_x, _base_mask, _comp_mask)
        _iv_c = q._iv_one(_x[_is_cycle], _y[_is_cycle])
        _iv_s = q._iv_one(_x[_is_single], _y[_is_single])

        _rows.append(
            {
                "feature_name": _base_name,
                "cn_desc": _cn,
                "logic_desc": _logic,
                "na_cnt": _na_cnt,
                "total_cnt": _total,
                "na_rate": round(_na_rate, 6),
                "nunique": _nunique,
                "corr_y": None if pd.isna(_corr) else round(float(_corr), 6),
                "iv": None if pd.isna(_iv) else round(float(_iv), 6),
                "psi_last2w_vs_firstw": None if pd.isna(_psi) else round(float(_psi), 6),
                "iv_cycle_pass": None if pd.isna(_iv_c) else round(float(_iv_c), 6),
                "iv_single_pass": None if pd.isna(_iv_s) else round(float(_iv_s), 6),
                "notebook": "第三板块衍生.ipynb",
            }
        )

    _quality = pd.DataFrame(_rows)
    QUALITY_XLSX.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(QUALITY_XLSX) as _w:
        _quality.to_excel(_w, sheet_name="report", index=False)
    print("[WRITE_EXCEL] block3 feature quality saved:", str(QUALITY_XLSX.resolve()))

